# __Chapter 7. Optimization__

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


### [Algorithm] 황금분할탐색법

In [ ]:
import numpy as np

def golden_section_search(func, a, b, tol=1e-6, max_iter=10_000, 
                          maximize=False, verbose=True):
    """
    Golden Section Search (1D)
    - [a,b] 구간에서 단봉 함수의 최소/최대 근사
    - 오차: Ea(%) = (2-phi)*|(b-a)/x*|*100, x*는 현재 반복에서 선택된 최적 후보점
    """
    if a >= b:
        raise ValueError("Require a < b for the initial interval.")

    # 최대화는 -f 최소화로 변환
    if maximize:
        def _f(x): return -func(x)
    else:
        _f = func

    phi = (1 + np.sqrt(5.0)) / 2.0
    inv_phi2 = (3 - np.sqrt(5.0)) / 2.0
    inv_phi  = 1 - inv_phi2

    c = a + inv_phi2 * (b - a)
    d = a + inv_phi  * (b - a)
    fc = _f(c)
    fd = _f(d)

    if verbose:
        print(f"{'iter':>4} | {'a':>10} | {'c':>10} | {'d':>10} | {'b':>10} | {'f(c)':>10} | {'f(d)':>10} | {'x*':>10} | {'Ea(%)':>10}")
        print("-"*105)

    it = 0
    while it < max_iter and (b - a) > tol:
        it += 1

        # 현재 반복에서 최적 후보점 선택
        if fc <= fd:
            x_star, f_star = c, fc
        else:
            x_star, f_star = d, fd

        Ea = (2 - phi) * abs((b - a)/max(abs(x_star), 1e-15)) * 100.0

        if verbose:
            if maximize:
                print(f"{it:4d} | {a:10.4f} | {c:10.4f} | {d:10.4f} | {b:10.4f} | {-fc:10.4f} | {-fd:10.4f} | {x_star:10.4f} | {Ea:10.4f}")
            else:
                print(f"{it:4d} | {a:10.4f} | {c:10.4f} | {d:10.4f} | {b:10.4f} | {fc:10.4f} | {fd:10.4f} | {x_star:10.4f} | {Ea:10.4f}")

        # 구간 갱신
        if fc <= fd:
            b, d, fd = d, c, fc
            c = a + inv_phi2 * (b - a)
            fc = _f(c)
        else:
            a, c, fc = c, d, fd
            d = a + inv_phi * (b - a)
            fd = _f(d)

        if Ea <= tol*100:
            break

    # 최종 최적점
    if fc <= fd:
        x_star, f_star = c, fc
    else:
        x_star, f_star = d, fd

    if maximize:
        f_star = -f_star

    return x_star, f_star, it


### [Algorithm] 2차 보간법

In [ ]:
import numpy as np

def quadratic_interpolation_search(func, a, b, c, tol_percent=0.01, max_iter=1000,
                                   maximize=False, verbose=True):
    """
    Quadratic (parabolic) interpolation search for 1D optimization.
    - 초기 3점 (a<b<c)에서 시작
    - 각 반복에서 2차 보간 꼭짓점 x_new를 계산
    - 수렴 기준: 근사 백분율 상대오차 ea% <= tol_percent
    - verbose=True이면 반복 과정을 표 형태로 출력

    Parameters
    ----------
    func : callable
        목적함수 f(x)
    a, b, c : float
        초기 3점 (a<b<c 권장)
    tol_percent : float
        근사 백분율 상대오차 허용치 (%)
    max_iter : int
        최대 반복 횟수
    maximize : bool
        True이면 최대화 문제
    verbose : bool
        True이면 반복 과정 표 출력

    Returns
    -------
    x_star : float
        추정 최적점
    f_star : float
        f(x_star)
    iters : int
        반복 횟수
    """
    if maximize:
        def _f(x): return -func(x)
    else:
        _f = func

    # 초기 함수값
    fa, fb, fc = _f(a), _f(b), _f(c)

    if verbose:
        print(f"{'iter':>4} | {'a':>10} | {'b':>10} | {'c':>10} | {'x_new':>10} | {'f(x_new)':>12} | {'ea(%)':>10}")
        print("-"*80)

    x_old = None
    ea = None
    it = 0

    while it < max_iter:
        it += 1
        # 포물선 꼭짓점 공식
        denom = (b - a) * (fb - fc) - (b - c) * (fb - fa)
        if abs(denom) < 1e-15 or not np.isfinite(denom):
            x_new = 0.5 * (a + c)  # 불안정 시 중점
        else:
            num = (b - a)**2 * (fb - fc) - (b - c)**2 * (fb - fa)
            x_new = b - 0.5 * (num / denom)

        f_new = _f(x_new)

        # 근사 백분율 상대오차
        if x_old is not None:
            ea = abs((x_new - x_old) / max(abs(x_new), 1e-15)) * 100.0
        else:
            ea = None

        if verbose:
            ea_str = f"{ea:10.4f}" if ea is not None else f"{'--':>10}"
            if maximize:
                print(f"{it:4d} | {a:10.4f} | {b:10.4f} | {c:10.4f} | {x_new:10.4f} | {-f_new:12.6f} | {ea_str}")
            else:
                print(f"{it:4d} | {a:10.4f} | {b:10.4f} | {c:10.4f} | {x_new:10.4f} | {f_new:12.6f} | {ea_str}")

        # 종료 판정
        if ea is not None and ea <= tol_percent:
            break

        # 갱신: f_new가 가장 작은 쪽을 b로 유지
        if x_new > b:
            if f_new < fb:
                a, fa = b, fb
                b, fb = x_new, f_new
            else:
                c, fc = x_new, f_new
        else:  # x_new < b
            if f_new < fb:
                c, fc = b, fb
                b, fb = x_new, f_new
            else:
                a, fa = x_new, f_new

        x_old = x_new

    # 최종 최적점 선택
    f_vals = [(fa, a), (fb, b), (fc, c)]
    f_star, x_star = min(f_vals, key=lambda t: t[0])
    if maximize:
        f_star = -f_star

    return x_star, f_star, it


### [Algorithm] 최대급경사법

In [ ]:
import numpy as np

def steepest_descent(func, grad, x0, alpha=0.1, tol=1e-6, max_iter=1000, 
                     maximize=False, verbose=True):
    """
    Steepest Descent Method (with fixed step size).
    - 1변수 및 다변수 함수 모두 지원
    
    Parameters
    ----------
    func : callable
        목적함수 f(x). x는 float 또는 ndarray.
    grad : callable
        Gradient 함수 ∇f(x). x는 float 또는 ndarray, 반환도 동일 타입.
    x0 : float or array_like
        초기값
    alpha : float
        step size (학습률).
    tol : float
        종료 기준 (gradient norm).
    max_iter : int
        최대 반복 횟수.
    maximize : bool
        True이면 최대화 문제. (gradient ascent)
    verbose : bool
        True이면 반복 과정을 표로 출력.

    Returns
    -------
    x_star : float or ndarray
        추정 최적점
    f_star : float
        f(x_star)
    iters : int
        반복 횟수
    """
    # 입력을 ndarray로 변환
    scalar_input = np.isscalar(x0)
    x = np.array([x0], dtype=float) if scalar_input else np.array(x0, dtype=float)
    x_old = None
    
    if verbose:
        header = f"{'iter':>4} | {'x':>20} | {'f(x)':>12} | {'||grad||':>12} | {'ea(%)':>10}"
        print(header)
        print("-"*80)
    
    for it in range(1, max_iter+1):
        g = grad(x[0]) if scalar_input else grad(x)
        g = np.array([g], dtype=float) if scalar_input else np.array(g, dtype=float)
        
        g_norm = np.linalg.norm(g)
        fval = func(x[0]) if scalar_input else func(x)
        
        # 근사 백분율 상대오차
        if x_old is not None:
            ea = np.linalg.norm(x - x_old) / max(np.linalg.norm(x), 1e-15) * 100.0
        else:
            ea = None
        
        if verbose:
            x_disp = f"{x[0]:.6f}" if scalar_input else str(x)
            ea_str = f"{ea:10.4f}" if ea is not None else f"{'--':>10}"
            print(f"{it:4d} | {x_disp:>20} | {fval:12.6f} | {g_norm:12.6e} | {ea_str}")
        
        # 종료 조건
        if g_norm < tol or (ea is not None and ea <= tol*100):
            break
        
        # 방향
        d = g if maximize else -g
        
        # 업데이트
        x_old = x.copy()
        x = x + alpha * d
    
    # 반환 시 스칼라/벡터 구분
    x_star = x[0] if scalar_input else x
    f_star = func(x_star) if scalar_input else func(x_star)
    return x_star, f_star, it
